In [10]:
import pandas as pd

df = pd.read_csv("bra_scores.csv")

score_cols = [col for col in df.columns if col.endswith("_Score")]

def get_best_score(row):
    scores = row[score_cols]
    best_score = scores.max()
    return best_score

def get_best_score_role(row):
    scores = row[score_cols]
    best_score = scores.max()
    best_score_col = scores.idxmax()
    role = best_score_col.replace("_Score", "")
    return role

def get_mean_score(row):
    scores = row[score_cols]
    mean_score = scores.mean()
    return mean_score

def get_median_score(row):
    scores = row[score_cols]
    median_score = scores.median()
    return median_score

def create_custom_score(row):
    max_score = row["Best_Score"]
    mean_score = row["Mean_Score"]
    median_score = row["Median_Score"]
    custom_score = (max_score * 0.5) + (mean_score * 0.25) + (median_score * 0.15) + row["Central_Midfielder_Score"] * 0.1
    return custom_score

df["Best_Score"] = df.apply(get_best_score, axis=1)
df["Best_Score_Role"] = df.apply(get_best_score_role, axis=1)
df["Mean_Score"] = df.apply(get_mean_score, axis=1)
df["Median_Score"] = df.apply(get_median_score, axis=1)
df["Custom_Score"] = df.apply(create_custom_score, axis=1)


In [11]:
pos_df = pd.read_csv("/home/giova/workspace/fm26/data/bra/interim/positions.csv")
demo_df = pd.read_csv("/home/giova/workspace/fm26/data/bra/interim/player_demographics.csv")[["Name", "Age"]]

merged_df = pd.merge(df[["Name", "Best_Score", "Best_Score_Role", "Custom_Score"]], pos_df, on="Name", how="left")
merged_df = pd.merge(merged_df, demo_df, on="Name", how="left")

def apply_age_penalty(row):
    if row["Age"] > 31:
        penalty = (row["Age"] - 32) * 0.2
        return row["Custom_Score"] - penalty
    else:
        return row["Custom_Score"]
    
merged_df["Custom_Score_with_Penalty"] = merged_df.apply(apply_age_penalty, axis=1)

In [12]:
goalkeepers = merged_df[merged_df["GK"]].sort_values(by="Custom_Score_with_Penalty", ascending=False)[["Name", "Best_Score", "Custom_Score_with_Penalty", "Best_Score_Role"]]
defenders = merged_df[merged_df["DR"] | merged_df["DL"] | merged_df["DC"]].sort_values(by="Custom_Score_with_Penalty", ascending=False)[["Name", "Best_Score", "Custom_Score_with_Penalty", "Best_Score_Role"]]
midfielders = merged_df[merged_df["DM"] | merged_df["MC"]].sort_values(by="Custom_Score_with_Penalty", ascending=False)[["Name", "Best_Score", "Custom_Score_with_Penalty", "Best_Score_Role"]]
attackers = merged_df[merged_df["ST"] | merged_df["AML"] | merged_df["AMR"] | merged_df["AMC"]].sort_values(by="Custom_Score_with_Penalty", ascending=False)[["Name", "Best_Score", "Custom_Score_with_Penalty", "Best_Score_Role"]]

In [19]:
attackers.head(50)

,Name,Best_Score,Custom_Score_with_Penalty,Best_Score_Role
231,Vinícius Júnior,16.540541,15.485594,Advanced_Forward
117,Joelinton,16.482759,15.463261,Wide_Target_Forward
161,Gabriel Martinelli,15.918919,15.019835,Winger
76,Gabriel Jesus,15.567568,15.010737,Winger
66,Evanilson,16.000000,14.872673,Anchor
46,David Neres,16.000000,14.856129,Advanced_Playmaker
211,Rodrygo,15.648649,14.717516,Advanced_Forward
224,Tetê,15.214286,14.608151,Inside_Wing_Back
166,Matheus Gonçalves,15.590909,14.597197,Winger
84,Galeno,15.538462,14.468851,Full_Back
